In [16]:
import argparse
import logging
import pickle as pkl
import cv2 as cv
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pandas as pd

In [17]:
def heatmap_on_image(heatmap, image, w=108, h=36):
    plt.figure(figsize=(w,h),dpi=1)
    hmax = sns.heatmap(heatmap, vmin=-1, vmax=1,
                    cmap="bwr",
                    xticklabels=False, yticklabels=False, cbar=False, 
                    alpha=1, zorder=1)
    # plt.savefig("os.path.join(relevance_maps_path, "heatmap.png"))
    hmax.imshow(image,
                cmap="gray",
                alpha=0.5,
                aspect = hmax.get_aspect(),
                extent = hmax.get_xlim() + hmax.get_ylim(),
                zorder = 2)
    plt.tight_layout()
    # plt.savefig("os.path.join(relevance_maps_path, "heatmap_background.png"))

    ax = plt.gca()
    canvas = ax.figure.canvas
    canvas.draw()
    hoi = np.frombuffer(canvas.tostring_rgb(), dtype='uint8')
    hoi = np.reshape(hoi, (h,w,3))
    hoi = cv.cvtColor(hoi,cv.COLOR_BGR2RGBA)
    plt.close()
    return hoi

In [18]:
def find_obj(obj_folder, sample, side):
    for root, folders, files in os.walk(obj_folder):
        for file in files:
            _, ext = os.path.splitext(file)
            if ext == ".obj":
                file_sample, file_side, _ = file.split("_")
                if sample == file_sample and side == file_side:
                    return os.path.join(os.path.abspath(root), file)

In [19]:
maps_path = "2D heatmaps"
data_path = "pubis_data_proc_25_panorama_seleccion_javi_gray_mask"
obj_paths = "pubis_seleccion_javi"

In [20]:
relevance_maps = {}
i = 0
for root, folders, files in os.walk(maps_path):
    for file in files:
        _, ext = os.path.splitext(file)
        if ext == ".pkl":
            file_path = os.path.join(root, file)

            sample, side, axis, _ = file.split("_")

            print(sample, side, axis, file_path)

            with open(file_path, 'rb') as f:
                data = pkl.load(f)

            img = np.array(data).T
            # img = cv.imread(file_path, 0) / 255

            img = cv.resize(img, (108, 36))
            # img = cv.resize(img, (540, 180))
            img_max = np.max(img)
            img = img / img_max
            img = np.where(img >= 0.75, img, 0)
            print(np.min(img), np.max(img))

            mask = cv.imread(os.path.join(data_path, "{}_{}".format(sample,side), "{}_{}_0_panorama_ext_{}_gray_mask_input.png".format(sample,side,axis)), cv.IMREAD_GRAYSCALE)
            # mask = cv.imread(os.path.join(data_path, "{}_{}".format(sample,side), "{}_{}_0_panorama_ext_{}_gray_mask.png".format(sample,side,axis)), cv.IMREAD_GRAYSCALE)

            mask = mask / 255.0
            img = img * mask
            img = img.astype('float32')

            img_orig = img.copy()
            relevance_map = img/2
            relevance_map[:,:36] += relevance_map[:,72:]
            relevance_map = relevance_map[:,:72]
            # relevance_map[:,:180] += relevance_map[:,360:]
            # relevance_map = relevance_map[:,:360]
            # cv.imshow("COMPLETE", relevance_map)

            cv.imwrite(os.path.join(root,"relevance_map_{}_{}_{}.png".format(sample,side,axis)), relevance_map*255)
            cv.imwrite(os.path.join(root,"relevance_map_{}_{}_{}_complete.png".format(sample,side,axis)), img_orig*255)
            relevance_maps_path = os.path.join(os.path.abspath(root),"relevance_map_{}_{}_{}.png".format(sample,side,axis))

            panorama = cv.imread(os.path.join(data_path, "{}_{}".format(sample,side), "{}_{}_0_panorama_ext_{}_gray_input.png".format(sample,side,axis)), cv.IMREAD_GRAYSCALE)
            # panorama = cv.imread(os.path.join(data_path, "{}_{}".format(sample,side), "{}_{}_0_panorama_ext_{}_gray.png".format(sample,side,axis)), cv.IMREAD_GRAYSCALE)
            heatmap = heatmap_on_image(img_orig, panorama, w=108, h=36)
            # heatmap = heatmap_on_image(img_orig, panorama, w=540, h=180)
            heatmap = cv.resize(heatmap, (540, 180))
            cv.imwrite(os.path.join(root,"heatmap_{}_{}_{}_complete.png".format(sample,side,axis)), heatmap)

            panorama = panorama[:,:72]
            # panorama = panorama[:,:360]
            heatmap = heatmap_on_image(relevance_map, panorama, w=108, h=36)
            # heatmap = heatmap_on_image(relevance_map, panorama, w=360, h=180)
            heatmap = cv.resize(heatmap, (360, 180))
            cv.imwrite(os.path.join(root,"heatmap_{}_{}_{}.png".format(sample,side,axis)), heatmap)

            if "resnet" in file_path:
                    net = "Resnet"
            else:
                net = "Panorama"

            obj = find_obj(obj_paths, sample, side)

            relevance_maps[i] = {
                "net": net,
                "sample" : sample,
                "side" : side,
                "axis" : axis,
                "obj" : obj,
                "relevance": relevance_maps_path,
                "max" : img_max

            }
            i += 1

            # cv.waitKey(0)
            # cv.destroyAllWindows()

relevance_maps_df = pd.DataFrame.from_dict(relevance_maps).T
relevance_maps_df.to_csv("relevance_maps.csv", sep=";", index=False)

367 Izq X 2D heatmaps/resnet(X,Y,Z)/367/367_Izq_X_2channels.pkl
0.0 1.0
367 Izq Z 2D heatmaps/resnet(X,Y,Z)/367/367_Izq_Z_2channels.pkl
0.0 1.0
367 Izq Y 2D heatmaps/resnet(X,Y,Z)/367/367_Izq_Y_2channels.pkl
0.0 1.0
36 Izq Z 2D heatmaps/resnet(X,Y,Z)/36/36_Izq_Z_2channels.pkl
0.0 1.0
36 Izq Y 2D heatmaps/resnet(X,Y,Z)/36/36_Izq_Y_2channels.pkl
0.0 1.0
36 Izq X 2D heatmaps/resnet(X,Y,Z)/36/36_Izq_X_2channels.pkl
0.0 1.0
141 Izq Z 2D heatmaps/resnet(X,Y,Z)/141/141_Izq_Z_2channels.pkl
0.0 1.0
141 Izq X 2D heatmaps/resnet(X,Y,Z)/141/141_Izq_X_2channels.pkl
0.0 1.0
141 Izq Y 2D heatmaps/resnet(X,Y,Z)/141/141_Izq_Y_2channels.pkl
0.0 1.0
341 Izq Z 2D heatmaps/resnet(X,Y,Z)/341/341_Izq_Z_2channels.pkl
0.0 1.0
341 Izq Y 2D heatmaps/resnet(X,Y,Z)/341/341_Izq_Y_2channels.pkl
0.0 1.0
341 Izq X 2D heatmaps/resnet(X,Y,Z)/341/341_Izq_X_2channels.pkl
0.0 1.0
14 Dch Y 2D heatmaps/resnet(X,Y,Z)/14/14_Dch_Y_2channels.pkl
0.0 1.0
14 Dch Z 2D heatmaps/resnet(X,Y,Z)/14/14_Dch_Z_2channels.pkl
0.0 1.0
14 Dch 